<a href="https://colab.research.google.com/github/mafloress/ProyectoFinalML1/blob/mlops-pipeline-setup/bank_marketing_project/training_pipeline/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrenamiento del Modelo para la Predicción de Marketing Bancario

## 1. Importar Bibliotecas Necesarias

In [2]:
# prompt: Instala LazyPredict

!pip install LazyPredict

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.5/720.5 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.0/119.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.9/194.9 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import joblib # Para guardar el modelo
import json # Para cargar nombres de características

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_auc_score

try:
    from lazypredict.Supervised import LazyClassifier
    lazypredict_available = True
except ImportError:
    print("LazyPredict no instalado. Omitiendo pasos de LazyPredict. Considera instalar con: pip install lazypredict")
    lazypredict_available = False

import mlflow
import mlflow.sklearn

# Mostrar gráficos en línea
%matplotlib inline

# Establecer estilo de gráficos
plt.style.use('seaborn-v0_8-whitegrid')

LazyPredict no instalado. Omitiendo pasos de LazyPredict. Considera instalar con: pip install lazypredict


ModuleNotFoundError: No module named 'mlflow'

## 2. Cargar Datos

In [ ]:
data_path = '../feature_pipeline/bank-features-selected.csv'
selected_features_path = '../feature_pipeline/selected_feature_names.json'
df = None
X = None
y = None
target_column = 'y' # Como se definió en la ingeniería de características
feature_names = []

if os.path.exists(data_path):
    try:
        df = pd.read_csv(data_path)
        print(f"Datos cargados desde '{data_path}'. Dimensiones: {df.shape}")

        if target_column in df.columns:
            X = df.drop(columns=[target_column])
            y = df[target_column]
            feature_names = X.columns.tolist()
            print(f"Dimensiones de Características (X): {X.shape}")
            print(f"Dimensiones del Objetivo (y): {y.shape}")
            print(f"Distribución del objetivo:\n{y.value_counts(normalize=True)}")

            if y.isnull().any():
                print(f"\nAdvertencia: La variable objetivo '{target_column}' contiene {y.isnull().sum()} valores NaN.")
                print("Eliminando filas con objetivo NaN para el entrenamiento del modelo.")
                nan_target_indices = y[y.isnull()].index
                X = X.drop(index=nan_target_indices)
                y = y.drop(index=nan_target_indices)
                feature_names = X.columns.tolist() # Actualizar nombres de características si se eliminaron filas
                print(f"Dimensiones de X limpias: {X.shape}, Dimensiones de y limpias: {y.shape}")
                print(f"Nueva distribución del objetivo después de eliminar NaN:\n{y.value_counts(normalize=True)}")

            if not pd.api.types.is_integer_dtype(y):
                print("\nAdvertencia: La variable objetivo no es de tipo entero. Intentando conversión...")
                try:
                    y = y.astype(int)
                    print("Variable objetivo convertida a entero exitosamente.")
                except ValueError as ve:
                    print(f"Error convirtiendo objetivo a int: {ve}. Verifica si hay valores no numéricos.")
                    X, y = None, None
        else:
            print(f"Error: Columna objetivo '{target_column}' no encontrada en el DataFrame cargado.")
            X, y = None, None

        # Opcionalmente cargar nombres de características desde JSON si X.columns no es suficiente (ej. si el orden importa y fue fijado)
        if os.path.exists(selected_features_path):
             with open(selected_features_path, 'r') as f:
                 feature_names_from_file = json.load(f)
                 if set(feature_names_from_file) == set(X.columns.tolist()):
                     feature_names = feature_names_from_file # Usar esto si el orden es crítico y definido en el archivo
                     print(f"Nombres de características cargados desde {selected_features_path}")
                 else:
                     print("Advertencia: Nombres de características desde JSON no coinciden con columnas en CSV. Usando columnas de CSV.")
                     feature_names = X.columns.tolist()
        else:
            print(f"Advertencia: {selected_features_path} no encontrado. Usando nombres de características de las columnas del CSV.")
            if X is not None: feature_names = X.columns.tolist()

    except Exception as e:
        print(f"Error cargando datos desde '{data_path}': {e}")
        df, X, y = None, None, None
else:
    print(f"Error: Archivo de datos no encontrado en '{data_path}'. Asegúrate de que el pipeline de ingeniería de características se haya ejecutado.")

## 3. Dividir Datos

In [ ]:
X_train, X_test, y_train, y_test = None, None, None, None

if X is not None and y is not None:
    if not y.empty and len(y.unique()) > 1:
        try:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
            print("Datos divididos en conjuntos de entrenamiento y prueba.")
            print(f"Dimensiones de X_train: {X_train.shape}, Dimensiones de y_train: {y_train.shape}")
            print(f"Dimensiones de X_test: {X_test.shape}, Dimensiones de y_test: {y_test.shape}")
        except ValueError as ve_split:
             print(f"Error durante la división de datos (posiblemente debido a muy pocas muestras para una clase para estratificar): {ve_split}")
             print("Intentando división sin estratificación...")
             try:
                 X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                 print("División de datos (sin estratificación) exitosa.")
             except Exception as e_split_nostrat:
                 print(f"Error durante la división de datos sin estratificación: {e_split_nostrat}")
        except Exception as e_split:
            print(f"Un error inesperado durante la división de datos: {e_split}")
    else:
        print("La variable objetivo 'y' está vacía o tiene solo una clase. No se puede realizar división estratificada o entrenamiento significativo.")
else:
    print("X o y no están disponibles. Omitiendo división de datos.")

## 4. Ejecutar LazyPredict (Opcional - para comparación)

In [ ]:
models_summary = None
if lazypredict_available and X_train is not None and y_train is not None and X_test is not None and y_test is not None:
    if X_train.isnull().sum().sum() > 0 or y_train.isnull().sum().sum() > 0:
        print("Advertencia: Se encontraron NaNs en los datos de entrenamiento. LazyPredict podría fallar o producir resultados no confiables.")

    print("Ejecutando LazyClassifier...")
    try:
        clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None, random_state=42)
        models, predictions = clf.fit(X_train, X_test, y_train, y_test)
        print("\nResultados de LazyClassifier:")
        display(models)
        models_summary = models
    except Exception as e:
        print(f"Error ejecutando LazyClassifier: {e}")
elif not lazypredict_available:
    print("LazyPredict no está instalado. Omitiendo este paso.")
else:
    print("Datos de entrenamiento/prueba no disponibles o no preparados adecuadamente. Omitiendo LazyClassifier.")

### Discusión de los Resultados de LazyPredict
LazyPredict proporciona una visión general rápida de varios modelos. Buscamos un F1-score, ROC AUC y Balanced Accuracy altos debido al desequilibrio de clases.

## 5. Configuración del Experimento MLflow

In [ ]:
experiment_name = "Bank Marketing Predictions"
try:
    experiment_id = mlflow.create_experiment(experiment_name)
    print(f"Experimento MLflow '{experiment_name}' creado con ID: {experiment_id}")
except mlflow.exceptions.MlflowException as e:
    if "already exists" in str(e):
        experiment = mlflow.get_experiment_by_name(experiment_name)
        experiment_id = experiment.experiment_id
        print(f"Experimento MLflow '{experiment_name}' ya existe con ID: {experiment_id}")
    else:
        raise e
mlflow.set_experiment(experiment_name=experiment_name)

## 6. Ajuste de Hiperparámetros y Seguimiento con MLflow

Seleccionaremos un modelo (por ejemplo, RandomForestClassifier o GradientBoostingClassifier, posiblemente guiados por los resultados de LazyPredict), definiremos una cuadrícula de parámetros y usaremos GridSearchCV para el ajuste. Los resultados se registrarán con MLflow.

In [ ]:
model_to_tune_name = "RandomForestClassifier" # Opción predeterminada, puede cambiarse
model_for_tuning = None
param_grid = {}

if models_summary is not None and not models_summary.empty:
    preferred_models_for_tuning = ["RandomForestClassifier", "GradientBoostingClassifier", "LogisticRegression"]
    # Ejemplo: Elegir basado en F1 Score de LazyPredict si está disponible
    if 'F1 Score' in models_summary.columns:
        sorted_lazy_models = models_summary.sort_values('F1 Score', ascending=False)
        for m_name in sorted_lazy_models.index:
            if m_name in preferred_models_for_tuning:
                model_to_tune_name = m_name
                print(f"'{model_to_tune_name}' seleccionado para ajuste basado en F1 Score de LazyPredict.")
                break
    elif 'ROC AUC' in models_summary.columns:
        sorted_lazy_models = models_summary.sort_values('ROC AUC', ascending=False)
        for m_name in sorted_lazy_models.index:
            if m_name in preferred_models_for_tuning:
                model_to_tune_name = m_name
                print(f"'{model_to_tune_name}' seleccionado para ajuste basado en ROC AUC de LazyPredict.")
                break
else:
    print(f"Resumen de LazyPredict no disponible o vacío. Usando por defecto '{model_to_tune_name}' para ajuste.")

print(f"\nConfigurando para ajuste de {model_to_tune_name}...")
if model_to_tune_name == "RandomForestClassifier":
    model_for_tuning = RandomForestClassifier(random_state=42, class_weight='balanced')
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }
elif model_to_tune_name == "GradientBoostingClassifier":
    model_for_tuning = GradientBoostingClassifier(random_state=42)
    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7]
    }
elif model_to_tune_name == "LogisticRegression":
    model_for_tuning = LogisticRegression(random_state=42, class_weight='balanced', solver='liblinear', max_iter=1000)
    param_grid = {
        'C': [0.1, 1.0, 10.0],
        'penalty': ['l1', 'l2']
    }
else:
    print(f"Ajuste para {model_to_tune_name} no predefinido. Usando RandomForestClassifier por defecto.")
    model_to_tune_name = "RandomForestClassifier"
    model_for_tuning = RandomForestClassifier(random_state=42, class_weight='balanced')
    param_grid = {
        'n_estimators': [50, 100], # Cuadrícula reducida para el caso por defecto
        'max_depth': [10, None]
    }

best_model = None
if X_train is not None and y_train is not None and model_for_tuning is not None:
    print(f"Iniciando GridSearchCV para {model_to_tune_name}...")
    # Usando 'f1_weighted' ya que es bueno para clases desequilibradas. 'roc_auc' también es una opción fuerte.
    grid_search = GridSearchCV(estimator=model_for_tuning, param_grid=param_grid, cv=3, scoring='f1_weighted', verbose=1, n_jobs=-1)
    try:
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        print(f"\nMejores parámetros para {model_to_tune_name}: {grid_search.best_params_}")
        print(f"Mejor F1_weighted score de GridSearchCV: {grid_search.best_score_:.4f}")
    except Exception as e:
        print(f"Error durante GridSearchCV: {e}")
else:
    print("Datos de entrenamiento o modelo para ajuste no disponibles. Omitiendo GridSearchCV.")

In [ ]:
if best_model is not None and X_test is not None and y_test is not None:
    with mlflow.start_run(run_name=f"Tuned {model_to_tune_name}") as run:
        run_id = run.info.run_id
        print(f"MLflow Run ID: {run_id}")
        mlflow.log_param("model_type", model_to_tune_name)
        mlflow.log_params(grid_search.best_params_)
        mlflow.log_param("features", feature_names) # Registrar lista de nombres de características

        # Entrenar modelo final con mejores parámetros (ya hecho por GridSearchCV si refit=True, que es el valor por defecto)
        # best_model.fit(X_train, y_train) # No necesario si GridSearchCV refit es True

        y_pred_tuned = best_model.predict(X_test)
        y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]

        # Registrar métricas
        accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
        f1_weighted_tuned = f1_score(y_test, y_pred_tuned, average='weighted')
        f1_macro_tuned = f1_score(y_test, y_pred_tuned, average='macro')
        f1_positive_class_tuned = f1_score(y_test, y_pred_tuned, pos_label=1) # F1 para la clase 'yes'
        precision_weighted_tuned = precision_score(y_test, y_pred_tuned, average='weighted')
        precision_macro_tuned = precision_score(y_test, y_pred_tuned, average='macro')
        recall_weighted_tuned = recall_score(y_test, y_pred_tuned, average='weighted')
        recall_macro_tuned = recall_score(y_test, y_pred_tuned, average='macro')
        roc_auc_tuned = roc_auc_score(y_test, y_pred_proba_tuned)

        mlflow.log_metric("accuracy", accuracy_tuned)
        mlflow.log_metric("f1_weighted", f1_weighted_tuned)
        mlflow.log_metric("f1_macro", f1_macro_tuned)
        mlflow.log_metric("f1_positive_class", f1_positive_class_tuned)
        mlflow.log_metric("precision_weighted", precision_weighted_tuned)
        mlflow.log_metric("precision_macro", precision_macro_tuned)
        mlflow.log_metric("recall_weighted", recall_weighted_tuned)
        mlflow.log_metric("recall_macro", recall_macro_tuned)
        mlflow.log_metric("roc_auc", roc_auc_tuned)

        print(f"\nRendimiento de {model_to_tune_name} Ajustado:")
        print(f"  Accuracy: {accuracy_tuned:.4f}")
        print(f"  F1-score (weighted): {f1_weighted_tuned:.4f}")
        print(f"  F1-score (macro): {f1_macro_tuned:.4f}")
        print(f"  F1-score (clase positiva 'yes'): {f1_positive_class_tuned:.4f}")
        print(f"  ROC AUC: {roc_auc_tuned:.4f}")

        # Registrar informe de clasificación como artefacto de archivo de texto
        report_str = classification_report(y_test, y_pred_tuned, target_names=['No (0)', 'Yes (1)'])
        with open("classification_report.txt", "w") as f:
            f.write(report_str)
        mlflow.log_artifact("classification_report.txt")
        print("\nInforme de Clasificación (Modelo Ajustado):")
        print(report_str)

        # Registrar gráfico de matriz de confusión
        fig_cm, ax_cm = plt.subplots(figsize=(6,4))
        cm_tuned = confusion_matrix(y_test, y_pred_tuned)
        sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
                    xticklabels=['Predicho No (0)', 'Predicho Yes (1)'],
                    yticklabels=['Real No (0)', 'Real Yes (1)'])
        ax_cm.set_xlabel('Etiqueta Predicha')
        ax_cm.set_ylabel('Etiqueta Real')
        ax_cm.set_title(f'Matriz de Confusión - {model_to_tune_name} Ajustado')
        mlflow.log_figure(fig_cm, "confusion_matrix_tuned.png")
        plt.show()

        # Registrar gráfico de importancia de características (si aplica)
        if hasattr(best_model, 'feature_importances_'):
            importances = best_model.feature_importances_
            feature_importance_df = pd.DataFrame({'feature': X_train.columns, 'importance': importances})
            feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False).head(15) # Top 15

            fig_fi, ax_fi = plt.subplots(figsize=(10, 6))
            sns.barplot(x='importance', y='feature', data=feature_importance_df, ax=ax_fi)
            ax_fi.set_title(f'Importancia de Características - {model_to_tune_name} Ajustado')
            plt.tight_layout()
            mlflow.log_figure(fig_fi, "feature_importances_tuned.png")
            plt.show()

        # Registrar el modelo
        mlflow.sklearn.log_model(best_model, f"tuned_{model_to_tune_name.lower()}_model")
        print(f"{model_to_tune_name} ajustado registrado en MLflow.")

        # Guardar el mejor modelo ajustado usando joblib también para uso directo
        best_model_filename = 'best_tuned_model.joblib'
        joblib.dump(best_model, best_model_filename)
        print(f"Mejor modelo ajustado guardado localmente como '{best_model_filename}'")

else:
    print("Mejor modelo no disponible del ajuste, o datos de prueba no disponibles. Omitiendo seguimiento MLflow y guardado final del modelo.")

### Discusión del Rendimiento del Modelo Ajustado

Compara las métricas del modelo ajustado (Accuracy, F1-scores, Precision, Recall, ROC AUC) con el modelo base entrenado en el paso 5 y los resultados de LazyPredict.
-   ¿Mejoró el ajuste de hiperparámetros el rendimiento, especialmente para la clase minoritaria ('yes')?
-   ¿Cómo se comparan el F1-score (clase positiva), ROC AUC y Balanced Accuracy?
-   El gráfico de importancia de características (si se generó) puede proporcionar información sobre qué impulsa las predicciones del modelo.

El modelo registrado en MLflow (`tuned_{model_name}_model`) y guardado localmente como `best_tuned_model.joblib` es ahora el candidato para el pipeline de inferencia.

## 7. Modelo Inicial (del paso 5) - Registro en MLflow (Opcional)

In [ ]:
# Esta sección es para el modelo entrenado en el paso 5 (antes del ajuste)
# Es útil si deseas registrar ese modelo inicial en MLflow para comparación
# Asumiendo que 'final_model' y 'model_name_to_train' son de la celda anterior (paso 5)

initial_model_object = None # Marcador de posición para el modelo del paso 5
initial_model_name = None # Marcador de posición

# Intentar cargar el modelo guardado previamente si existe de la lógica del paso 5
# Esto requiere saber qué modelo se guardó. Asumamos que se basó en la variable 'model_name_to_train' de esa celda.
# Las variables 'final_model' y 'model_name_to_train' de la celda de la Sección 5 podrían sobrescribirse o estar fuera de alcance.
# Por simplicidad, redefiniremos o nos aseguraremos de que estén disponibles si esta celda se ejecuta de forma independiente.
# Si este notebook se ejecuta de arriba hacia abajo, 'final_model' y 'model_name_to_train' *podrían* aún mantener los valores de la sección 5.
# Sin embargo, la variable 'model_name_to_train' fue reutilizada para el ajuste. Asumamos que el primer modelo entrenado se guardó con un nombre genérico o su nombre específico.

# Para hacer esta celda ejecutable y demostrar el registro para el *primer* modelo (antes del ajuste):
# Idealmente, volveríamos a ejecutar parte del paso 5 o cargaríamos su artefacto guardado.
# Por ahora, asumamos que 'final_model' y 'model_name_to_train' de ese entrenamiento inicial son accesibles.
# Si la variable 'final_model' del paso 5 no está disponible aquí, esta celda necesitaría recargarla o reentrenarla.

# Ejemplo: si el primer modelo entrenado fue RandomForestClassifier y se guardó como 'randomforestclassifier_model.joblib'
first_model_path = None
if 'model_name_to_train' in locals() and os.path.exists(f'{model_name_to_train.lower()}_model.joblib'):
    first_model_path = f'{model_name_to_train.lower()}_model.joblib'
    initial_model_name = model_name_to_train # Este es en realidad el nombre del modelo elegido en el paso 5
elif os.path.exists('randomforestclassifier_model.joblib'): # Un valor predeterminado común
    first_model_path = 'randomforestclassifier_model.joblib'
    initial_model_name = 'RandomForestClassifier'

if first_model_path and os.path.exists(first_model_path):
    print(f"Cargando modelo inicial '{initial_model_name}' desde {first_model_path} para registro en MLflow...")
    initial_model_object = joblib.load(first_model_path)

    if initial_model_object and X_test is not None and y_test is not None:
        with mlflow.start_run(run_name=f"Initial {initial_model_name}") as run_initial:
            print(f"MLflow Run ID para modelo inicial: {run_initial.info.run_id}")
            mlflow.log_param("model_type", initial_model_name)
            mlflow.log_param("parameters", initial_model_object.get_params())
            mlflow.log_param("features", feature_names)

            y_pred_initial = initial_model_object.predict(X_test)
            y_pred_proba_initial = initial_model_object.predict_proba(X_test)[:, 1]

            accuracy_initial = accuracy_score(y_test, y_pred_initial)
            f1_weighted_initial = f1_score(y_test, y_pred_initial, average='weighted')
            f1_macro_initial = f1_score(y_test, y_pred_initial, average='macro')
            f1_pos_initial = f1_score(y_test, y_pred_initial, pos_label=1)
            roc_auc_initial = roc_auc_score(y_test, y_pred_proba_initial)

            mlflow.log_metric("accuracy", accuracy_initial)
            mlflow.log_metric("f1_weighted", f1_weighted_initial)
            mlflow.log_metric("f1_macro", f1_macro_initial)
            mlflow.log_metric("f1_positive_class", f1_pos_initial)
            mlflow.log_metric("roc_auc", roc_auc_initial)

            print(f"\nRendimiento del {initial_model_name} Inicial (para registro en MLflow):")
            print(f"  Accuracy: {accuracy_initial:.4f}, F1 (weighted): {f1_weighted_initial:.4f}, ROC AUC: {roc_auc_initial:.4f}")

            mlflow.sklearn.log_model(initial_model_object, f"initial_{initial_model_name.lower()}_model")
            print(f"{initial_model_name} inicial registrado en MLflow.")
    else:
        print("Modelo inicial del paso 5 no disponible o datos de prueba faltantes. Omitiendo registro en MLflow para él.")
else:
    print("Ruta al modelo inicial del paso 5 no determinada o archivo no existe. Omitiendo registro en MLflow para él.")